# 02_silver_transform

Casts Bronze's raw types into business-ready types (Requirement 2), applies
quality rules, and performs **idempotent MERGE** upserts into `silver_klines`
and `dim_symbol` (Requirement 3), logging every run (Requirement 4).

In [ ]:
import uuid
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, TimestampType,
    DecimalType, IntegerType
)
from pyspark.sql.functions import (
    col, lit, current_timestamp, max as spark_max, get_json_object,
    from_json, explode
)
from delta.tables import DeltaTable

## Constants & schemas (same table names / log schema as Bronze)

In [ ]:
CATALOG = "workspace"
SCHEMA = "cryptolens"

BRONZE_KLINES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_klines"
BRONZE_EXCHANGE_INFO_TABLE = f"{CATALOG}.{SCHEMA}.bronze_exchange_info"
SILVER_KLINES_TABLE = f"{CATALOG}.{SCHEMA}.silver_klines"
DIM_SYMBOL_TABLE = f"{CATALOG}.{SCHEMA}.dim_symbol"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("parameter_processed", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
])

## Audit logging helper (Requirement 4) — identical to Bronze's

In [ ]:
def log_execution(spark, layer, parameter_processed, load_type, start_time,
                   status, rows_inserted=0, rows_updated=0, error_message=None):
    end_time = datetime.utcnow()
    row = [(
        str(uuid.uuid4()), layer, parameter_processed, load_type,
        start_time, end_time, status,
        int(rows_inserted) if rows_inserted is not None else 0,
        int(rows_updated) if rows_updated is not None else 0,
        error_message,
    )]
    log_df = spark.createDataFrame(row, schema=LOG_SCHEMA)
    log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)

## Parameters
Same pattern as Bronze, so Silver can be backfilled independently for any
historical batch already sitting in Bronze. No symbol widget here either --
the driver loop below picks up every symbol already present in
bronze_klines for the given interval/load_type, so there's nothing to type
in manually and nothing that can silently resolve to an empty list.

In [ ]:
dbutils.widgets.text("interval", "1m")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"])

interval = dbutils.widgets.get("interval")
load_type = dbutils.widgets.get("load_type")

## silver_klines — strict casting + quality rules + idempotent MERGE

In [ ]:
def transform_silver_klines(spark, symbol, interval, load_type):
    start_time = datetime.utcnow()
    param_label = f"{symbol}-{interval}-{load_type}"
    try:
        bronze_df = (spark.table(BRONZE_KLINES_TABLE)
                     .filter((col("symbol") == symbol) &
                             (col("interval") == interval) &
                             (col("load_type") == load_type)))

        if bronze_df.rdd.isEmpty():
            log_execution(spark, "Bronze-to-Silver", param_label, load_type, start_time,
                          "SKIPPED", error_message="No matching Bronze rows")
            return

        # Requirement 2: explicit casting as data moves Bronze -> Silver.
        silver_df = (bronze_df
            .withColumn("open_time", (col("open_time") / 1000).cast(TimestampType()))
            .withColumn("close_time", (col("close_time") / 1000).cast(TimestampType()))
            .withColumn("open", col("open").cast(DecimalType(18, 8)))
            .withColumn("high", col("high").cast(DecimalType(18, 8)))
            .withColumn("low", col("low").cast(DecimalType(18, 8)))
            .withColumn("close", col("close").cast(DecimalType(18, 8)))
            .withColumn("volume", col("volume").cast(DecimalType(18, 8)))
            .withColumn("quote_asset_volume", col("quote_asset_volume").cast(DecimalType(18, 8)))
            .withColumn("number_of_trades", col("number_of_trades").cast(IntegerType()))
            .withColumn("taker_buy_base_vol", col("taker_buy_base_asset_volume").cast(DecimalType(18, 8)))
            .withColumn("taker_buy_quote_vol", col("taker_buy_quote_asset_volume").cast(DecimalType(18, 8)))
            .withColumn("load_timestamp", current_timestamp())
            .select("symbol", "interval", "open_time", "close_time", "open", "high", "low",
                    "close", "volume", "quote_asset_volume", "number_of_trades",
                    "taker_buy_base_vol", "taker_buy_quote_vol", "load_timestamp")
            # Quality quarantine rules -- drop rows that fail basic OHLC sanity
            # instead of letting bad data corrupt downstream aggregates.
            .filter(col("high") >= col("low"))
            .filter(col("close").between(col("low"), col("high")))
            .filter(col("open").between(col("low"), col("high")))
            .dropDuplicates(["symbol", "interval", "open_time"])  # belt-and-braces before MERGE
        )

        # Idempotent MERGE (Requirement 3): natural key = symbol + interval + open_time.
        # whenMatchedUpdateAll handles the case where a daily incremental file
        # later gets corrected by the following month's full-load file.
        if spark.catalog.tableExists(SILVER_KLINES_TABLE):
            target = DeltaTable.forName(spark, SILVER_KLINES_TABLE)
            (target.alias("t")
                .merge(silver_df.alias("s"),
                       "t.symbol = s.symbol AND t.interval = s.interval AND t.open_time = s.open_time")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
        else:
            silver_df.write.format("delta").saveAsTable(SILVER_KLINES_TABLE)

        row_count = silver_df.count()
        log_execution(spark, "Bronze-to-Silver", param_label, load_type, start_time,
                      "SUCCESS", rows_updated=row_count)
        print(f"[OK] {param_label} -> {row_count} rows merged into silver_klines")

    except Exception as e:
        log_execution(spark, "Bronze-to-Silver", param_label, load_type, start_time,
                      "FAILURE", error_message=str(e))
        print(f"[FAIL] {param_label}: {e}")
        raise

## Driver loop — processes every symbol Bronze actually has for this interval/load_type

In [ ]:
if not spark.catalog.tableExists(BRONZE_KLINES_TABLE):
    raise RuntimeError(
        "bronze_klines does not exist yet -- run 01_bronze_ingest first "
        "for this interval/load_type before running Silver."
    )

symbols = [r.symbol for r in (spark.table(BRONZE_KLINES_TABLE)
           .filter((col("interval") == interval) & (col("load_type") == load_type))
           .select("symbol").distinct().collect())]

if not symbols:
    print(f"[WARN] No bronze_klines rows found for interval={interval}, load_type={load_type}.")
else:
    print(f"Found {len(symbols)} symbol(s) in bronze_klines for interval={interval}, load_type={load_type}")

for sym in symbols:
    transform_silver_klines(spark, sym, interval, load_type)

## dim_symbol — real MERGE: insert (new listing), update (status/filter
change), soft-delete (delisted). This is what gives the pipeline a genuine
upsert to demonstrate, since kline data itself is append-only.

In [ ]:
def refresh_dim_symbol(spark):
    start_time = datetime.utcnow()
    try:
        if not spark.catalog.tableExists(BRONZE_EXCHANGE_INFO_TABLE):
            log_execution(spark, "Bronze-to-Silver", "dim_symbol_refresh", "incremental",
                          start_time, "SKIPPED", error_message="No exchangeInfo snapshots landed yet")
            return

        latest_ts = (spark.table(BRONZE_EXCHANGE_INFO_TABLE)
                     .agg(spark_max("load_timestamp").alias("ts")).collect()[0]["ts"])

        latest_snapshot = (spark.table(BRONZE_EXCHANGE_INFO_TABLE)
                            .filter(col("load_timestamp") == latest_ts))

        # Flatten the raw exchangeInfo JSON (one symbol object per array
        # element) into rows using get_json_object -- explicit field
        # extraction, no schema inference on the JSON itself.
        symbols_array = latest_snapshot.selectExpr(
            "explode(from_json(get_json_object(raw_json, '$.symbols'), "
            "'array<string>')) as symbol_json"
        )
        staging = symbols_array.select(
            get_json_object(col("symbol_json"), "$.symbol").alias("symbol"),
            get_json_object(col("symbol_json"), "$.baseAsset").alias("base_asset"),
            get_json_object(col("symbol_json"), "$.quoteAsset").alias("quote_asset"),
            get_json_object(col("symbol_json"), "$.status").alias("status"),
            get_json_object(col("symbol_json"), "$.isSpotTradingAllowed").cast("boolean").alias("is_spot_trading_allowed"),
        ).withColumn("load_timestamp", current_timestamp())

        if spark.catalog.tableExists(DIM_SYMBOL_TABLE):
            target = DeltaTable.forName(spark, DIM_SYMBOL_TABLE)
            (target.alias("t")
                .merge(staging.alias("s"), "t.symbol = s.symbol")
                .whenMatchedUpdate(
                    condition="t.status <> s.status OR t.is_spot_trading_allowed <> s.is_spot_trading_allowed",
                    set_={
                        "status": "s.status",
                        "is_spot_trading_allowed": "s.is_spot_trading_allowed",
                        "is_active": "true",
                        "load_timestamp": "s.load_timestamp",
                    })
                .whenNotMatchedInsert(values={
                    "symbol": "s.symbol", "base_asset": "s.base_asset", "quote_asset": "s.quote_asset",
                    "status": "s.status", "is_spot_trading_allowed": "s.is_spot_trading_allowed",
                    "is_active": "true", "load_timestamp": "s.load_timestamp",
                })
                # Soft-delete: a symbol present in dim_symbol but missing from
                # the latest snapshot is treated as delisted, not hard-deleted,
                # so history is preserved.
                .whenNotMatchedBySourceUpdate(set_={"status": "'DELISTED'", "is_active": "false"})
                .execute())
        else:
            (staging.withColumn("is_active", lit(True))
                    .write.format("delta").saveAsTable(DIM_SYMBOL_TABLE))

        row_count = staging.count()
        log_execution(spark, "Bronze-to-Silver", "dim_symbol_refresh", "incremental",
                      start_time, "SUCCESS", rows_updated=row_count)
        print(f"[OK] dim_symbol refreshed from snapshot {latest_ts} -> {row_count} symbols")

    except Exception as e:
        log_execution(spark, "Bronze-to-Silver", "dim_symbol_refresh", "incremental",
                      start_time, "FAILURE", error_message=str(e))
        print(f"[FAIL] dim_symbol refresh: {e}")
        raise

refresh_dim_symbol(spark)